In [1]:
import torch
from cnnmlp import CNNMLP


# Example configs
cnn_config = {
    "input_channel_num": 1,
    "channels_list": [8,8],
    "kernel_size": [3,3],
    "stride": 1,
    "dropout": 0.0,
    "norm_type": "batch",
    "num_groups": 4,
    "input_map_size": 1024  # Suppose the input image is 32x32
}

mlp_config = {
    "hidden_layers": [200,],
    "output_size": 72,
    "dropout": 0.5
}

# Instantiate model
model = CNNMLP(cnn_config, mlp_config)

# Make dummy input (batch_size=4, channels=1, height=32, width=32)
dummy_input = torch.randn(4, 1, 32, 32)

# Pass through the model
output = model(dummy_input)

print("Output shape:", output.shape)  # Should be [4, 2]
print(model)

Initialized CNNMLP with batch norm
Output shape: torch.Size([4, 72])
CNNMLP(
  (cnn): CNN(
    (network): Sequential(
      (0): CNN_one(
        (network): Sequential(
          (0): PeriodicConv2d(
            (conv): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1))
          )
          (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU()
          (3): Dropout2d(p=0.0, inplace=False)
        )
      )
      (1): CNN_one(
        (network): Sequential(
          (0): PeriodicConv2d(
            (conv): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1))
          )
          (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU()
          (3): Dropout2d(p=0.0, inplace=False)
        )
      )
    )
  )
  (mlp): MLP(
    (network): Sequential(
      (0): Linear(in_features=8192, out_features=200, bias=True)
      (1): LayerNorm((200,), eps=1e-05, elementwise_affine=True)
      (2): ReLU

In [6]:
for name, module in model.named_modules():
    if isinstance(module, torch.nn.BatchNorm2d):
        print(f"Layer: {name}")
        print(f"  Gamma (weight): {module.weight.data.shape}")
        print(f"  Beta (bias): {module.bias.data.shape}")
        break 

Layer: cnn.network.0.network.1
  Gamma (weight): torch.Size([8])
  Beta (bias): torch.Size([8])


In [3]:
import torch
import matplotlib.pyplot as plt

synth_data = torch.zeros(1, 1, 10, 10)
synth_data[0, 0, 0, -1] = 1.0

print("Synthetic Data (Top-Right corner is 1):")
print(synth_data[0, 0])

Synthetic Data (Top-Right corner is 1):
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])


In [4]:
from cnn import PeriodicConv2d
p_conv = PeriodicConv2d(in_channels=1, out_channels=1, kernel_size=3, padding=1)

torch.nn.init.constant_(p_conv.conv.weight, 1.0)
torch.nn.init.constant_(p_conv.conv.bias, 0.0)

output = p_conv(synth_data)
output_map = output[0, 0].detach().numpy()

print(output_map)

[[1. 0. 0. 0. 0. 0. 0. 0. 1. 1.]
 [1. 0. 0. 0. 0. 0. 0. 0. 1. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [7]:
import torch
from unet import UNet_A


# Example configs
cnn_config = {
    "input_channel_num": 1,
    "num_filters_enc": 8,
    "kernel_size": [3,3],
    "stride": 1,
    "dropout": 0.0,
    "norm_type": "batch",
    "num_groups": 4,
    "input_map_size": 1024  # Suppose the input image is 32x32
}

mlp_config = {
    "hidden_layers": [200,],
    "output_size": 72,
    "dropout": 0.5
}

# Instantiate model
model = UNet_A(cnn_config, mlp_config)

# Make dummy input (batch_size=4, channels=1, height=32, width=32)
dummy_input = torch.randn(4, 1, 32, 32)

# Pass through the model
output = model(dummy_input)

print("Output shape:", output.shape)  # Should be [4, 2]
print(model)

Output shape: torch.Size([4, 72])
UNet_A(
  (hid1): CNN_one(
    (network): Sequential(
      (0): PeriodicConv2d(
        (conv): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1))
      )
      (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout2d(p=0.0, inplace=False)
    )
  )
  (hid2): CNN_one(
    (network): Sequential(
      (0): PeriodicConv2d(
        (conv): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1))
      )
      (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout2d(p=0.0, inplace=False)
    )
  )
  (hid3): CNN_one(
    (network): Sequential(
      (0): PeriodicConv2d(
        (conv): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1))
      )
      (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout2d(p=0.0, inplace=False)
    )
  )
  (hid4): CNN_one(
    (network): Sequen

In [8]:
for name, module in model.named_modules():
    if isinstance(module, torch.nn.BatchNorm2d):
        print(f"Layer: {name}")
        print(f"  Gamma (weight): {module.weight.data.shape}")
        print(f"  Beta (bias): {module.bias.data.shape}")
        break 

Layer: hid1.network.1
  Gamma (weight): torch.Size([8])
  Beta (bias): torch.Size([8])
